# MAV Trim Calculations: Steady Flight and Numerical Solution

This document derives the trim calculation implemented by `control/trim.py`. Inputs are taken as climb angle, turn radius, and air speed. Returns control surface deflections and 12 MAV states

The solver does not simplify the force and moment model. Instead, it evaluates the full
nonlinear `Dynamics.derivative()` function and searches for a state and control vector
whose derivative matches steady flight.

---



## 1. What trim means

An aircraft is trimmed when its dynamic variables remain constant without changing the
control inputs. Straight-and-level flight is one trim condition, but a constant climb or
a steady turn can also be trimmed.

For the state

```math
\mathbf x = [p_n,p_e,p_d,u,v,w,\phi,\theta,\psi,p,q,r]^T,
```

steady flight does **not** generally mean $\dot{\mathbf x}=0$. Position changes as the
aircraft moves, and heading changes during a turn. Trim instead requires

```math
\dot{\mathbf x}=\dot{\mathbf x}_{\text{desired}},
```

where all unwanted accelerations and attitude-rate changes are zero.

---



## 2. Target quantities and sign conventions

`TrimTarget` specifies three quantities:

| Symbol | Code field | Meaning |
|---|---|---|
| $V_a$ | `airspeed` | Airspeed magnitude in m/s |
| $\gamma$ | `climb_angle_deg` | Flight-path angle, positive in a climb |
| $R$ | `turn_radius` | Signed turn radius; $R=\infty$ means straight flight |

The simulation uses North-East-Down coordinates, so positive climb produces a negative
down-position rate. The target inertial position rates are

```math
\dot p_n=V_a\cos\gamma,\qquad
\dot p_e=0,\qquad
\dot p_d=-V_a\sin\gamma.
```

This formulation initializes the heading from `parameters.state0` and expresses the
instantaneous flight path along North.

---



## 3. Optimization variables

The solver varies seven quantities:

```math
\mathbf z =
[\alpha,\beta,\phi,\delta_e,\delta_t,\delta_a,\delta_r]^T.
```

Here $\alpha$ is angle of attack, $\beta$ is sideslip, $\phi$ is bank angle, and the
four $\delta$ terms are elevator, throttle, aileron, and rudder. Pitch angle and body
rates are derived from these variables and the requested trajectory rather than optimized
independently. This keeps every trial point kinematically consistent.

The bounds used in `solve_trim()` are

```math
-0.7\leq\alpha\leq0.7,\quad
-0.4\leq\beta\leq0.4,\quad
-1.4\leq\phi\leq1.4,
```

with signed surface commands in $[-1,1]$ and throttle in $[0,1]$.

---



## 4. Airspeed resolved into body axes

With no separate wind state in this trim calculation, the air-relative velocity is resolved
into body components using $V_a$, $\alpha$, and $\beta$:

```math
u=V_a\cos\alpha\cos\beta,\qquad
v=V_a\sin\beta,\qquad
w=V_a\sin\alpha\cos\beta.
```

These equations preserve the requested speed because

```math
u^2+v^2+w^2=V_a^2.
```

The pitch attitude follows from angle of attack and flight-path angle:

```math
\theta=\alpha+\gamma.
```

Thus, a positive angle of attack requires the nose to be above the velocity direction.

---



## 5. Steady-turn angular rates

For a constant-radius turn, the desired heading rate is

```math
\dot\psi=\frac{V_a}{R}.
```

An infinite radius gives $\dot\psi=0$. In steady flight, bank and pitch are constant, so
$\dot\phi=\dot\theta=0$. The 3-2-1 Euler-rate relationship can be inverted to give
the body rates

```math
\begin{bmatrix}p\\q\\r\end{bmatrix}
=
\begin{bmatrix}
-\sin\theta\\
\sin\phi\cos\theta\\
\cos\phi\cos\theta
\end{bmatrix}\dot\psi.
```

The aircraft can therefore have nonzero body rates while its roll and pitch attitudes remain
constant.

---



## 6. Desired state derivative

Combining the trajectory constraints gives

```math
\dot{\mathbf x}_{\text{desired}}=
[V_a\cos\gamma,0,-V_a\sin\gamma,0,0,0,0,0,V_a/R,0,0,0]^T.
```

The zeros impose constant body velocity, constant roll and pitch, and constant body angular
rates. `Dynamics.derivative()` computes the actual derivative from the complete force,
moment, gravity, kinematic, and rigid-body equations.

For straight flight, interpret $V_a/R$ as zero. For level flight, $\gamma=0$, so the only
nonzero desired derivative is $\dot p_n=V_a$.

---



## 7. Residual and scaling

Define the raw trim error as

```math
\mathbf e(\mathbf z)=f(\mathbf x(\mathbf z),\boldsymbol\delta(\mathbf z))
-\dot{\mathbf x}_{\text{desired}},
```

where $f$ is `Dynamics.derivative()`. Position rates may naturally be tens of meters per
second, while attitude rates are measured in radians per second. Directly minimizing the raw
vector would let the numerically larger units dominate. The code therefore uses

```math
\mathbf r(\mathbf z)=S^{-1}\mathbf e(\mathbf z),
```

with

```math
S=\operatorname{diag}(20,20,20,10,10,10,1,1,1,10,10,10).
```

The reported `residual_norm` is $\|\mathbf r\|_2$, so it is a scaled convergence metric
rather than a physical quantity with one unit.

---



## 8. Finite-difference Jacobian

The solver needs the sensitivity of all 12 residuals to the seven optimization variables.
It approximates this $12\times7$ Jacobian one column at a time with a forward difference:

```math
J_{:,j}\approx
\frac{\mathbf r(\mathbf z+h\mathbf e_j)-\mathbf r(\mathbf z)}{h},
\qquad h=10^{-5}.
```

Each column requires another evaluation of the nonlinear aircraft dynamics. Finite
differences avoid deriving and maintaining analytical derivatives of the aerodynamic and
propulsion models.

---



## 9. Damped Gauss-Newton update

Near the current iterate, linearize the residual:

```math
\mathbf r(\mathbf z+\Delta\mathbf z)\approx
\mathbf r(\mathbf z)+J\Delta\mathbf z.
```

The code computes a Levenberg-Marquardt-style step by solving

```math
(J^TJ+\lambda I)\Delta\mathbf z=-J^T\mathbf r.
```

The damping $\lambda$ regularizes poorly conditioned directions. After applying the
variable bounds, an improving candidate is accepted and $\lambda$ is halved. A rejected
candidate leaves the variables unchanged and multiplies $\lambda$ by ten, making the next
step more conservative. Iteration stops when $\|\mathbf r\|_2$ is below the tolerance or
when `max_iterations` is reached.

---



## 10. Physical interpretation of the solution

At convergence, the four controls balance the forces and moments required by the chosen
trajectory:

- elevator and angle of attack establish the lift and pitching-moment balance,
- throttle balances drag and the longitudinal component of weight,
- bank angle supplies the lateral acceleration needed for a turn, and
- aileron and rudder balance roll/yaw moments and sideslip effects.

For the ideal level coordinated-turn approximation, force balance suggests

```math
\tan\phi\approx\frac{V_a^2}{gR}.
```

This is a useful reasonableness check, not an equation imposed by the solver. The actual bank
angle comes from the full nonlinear model and may differ because of sideslip, aerodynamic
coupling, climb angle, and the model's force coefficients.

---



## 11. Verification checklist

After computing a trim solution, verify:

1. `residual_norm` is below the requested tolerance.
2. No variable is unexpectedly pinned to a bound.
3. $\sqrt{u^2+v^2+w^2}$ equals the requested airspeed.
4. $\theta-\alpha$ equals the requested climb angle.
5. Straight flight has nearly zero $p$, $q$, and $r$.
6. A finite-radius turn has $\dot\psi\approx V_a/R$.
7. A short simulation initialized from the solution does not immediately accelerate away.

A small residual shows consistency with the implemented model; it does not by itself prove
that the aerodynamic model or chosen target is physically realistic.
